In [1]:
import pandas as pd
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt
import dagshub
import mlflow



In [2]:
df = pd.read_csv('../data/raw/smartphones_cleaned_v6.csv')
df.head()

,brand_name,model,price,rating,has_5g,has_nfc,has_ir_blaster,processor_brand,num_cores,processor_speed,...,refresh_rate,num_rear_cameras,num_front_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,extended_upto,resolution_width,resolution_height
0,oneplus,OnePlus 11 5G,54999,89.0,True,True,False,snapdragon,8.0,3.2,...,120,3,1.0,android,50.0,16.0,0,NaN,1440,3216
1,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,True,False,False,snapdragon,8.0,2.2,...,120,3,1.0,android,64.0,16.0,1,1024.0,1080,2412
2,samsung,Samsung Galaxy A14 5G,16499,75.0,True,False,False,exynos,8.0,2.4,...,90,3,1.0,android,50.0,13.0,1,1024.0,1080,2408
3,motorola,Motorola Moto G62 5G,14999,81.0,True,False,False,snapdragon,8.0,2.2,...,120,3,1.0,android,50.0,16.0,1,1024.0,1080,2400
4,realme,Realme 10 Pro Plus,24999,82.0,True,False,False,dimensity,8.0,2.6,...,120,3,1.0,android,108.0,16.0,0,NaN,1080,2412


In [3]:
df1 = df.copy()

In [4]:
# removing the unwanted columns
df1.drop(columns=['model'],inplace = True)

In [5]:
value_counts = df1['processor_brand'].value_counts()

# Keep brands with at least 10 observations
common_brands = value_counts[value_counts >= 10].index

df1['processor_brand'] = df1['processor_brand'].apply(
    lambda x: x if x in common_brands else 'other'
)


In [6]:
value_counts = df1['brand_name'].value_counts()

# Keep brands with at least 10 observations
common_brands = value_counts[value_counts >= 10].index

df1['brand_name'] = df1['brand_name'].apply(
    lambda x: x if x in common_brands else 'other'
)


In [7]:
df1.isnull().sum()

brand_name                     0
price                          0
rating                       101
has_5g                         0
has_nfc                        0
has_ir_blaster                 0
processor_brand                0
num_cores                      6
processor_speed               42
battery_capacity              11
fast_charging_available        0
fast_charging                211
ram_capacity                   0
internal_memory                0
screen_size                    0
refresh_rate                   0
num_rear_cameras               0
num_front_cameras              4
os                            14
primary_camera_rear            0
primary_camera_front           5
extended_memory_available      0
extended_upto                480
resolution_width               0
resolution_height              0
dtype: int64

In [ ]:
mlflow.set_experiment('exp-2')
with mlflow.start_run(run_name='exp-2 with null value imputation'):

        # getting important libraries
        from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,RobustScaler
        from sklearn.experimental import enable_iterative_imputer
        from sklearn.impute import SimpleImputer,KNNImputer,IterativeImputer
        from sklearn.pipeline import Pipeline
        from sklearn.compose import ColumnTransformer
        from sklearn.linear_model import LinearRegression
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.tree import DecisionTreeRegressor
        import xgboost as xgb
        from xgboost import XGBRegressor
        from sklearn.metrics import r2_score
        import lightgbm
        from lightgbm import LGBMRegressor

        # Separate features and target
        X = df1.drop('price', axis=1)
        y = df1['price']

        # sepearing nominal and numerical data 
        num_cols = X.select_dtypes(include=['int64','float64']).columns
        ord_cols = X.select_dtypes(include=['object']).columns

        # building pipeline

        num_simple_pipe = Pipeline([
            ('imputer',SimpleImputer()),
            ('scaler',RobustScaler())
        ])

        num_knn_pipe = Pipeline([
            ('imputer',KNNImputer()),
            ('scaler',RobustScaler())
        ])

        num_iter_pipe = Pipeline([
            ('imputer',IterativeImputer()),
            ('scaler',RobustScaler())
        ])


        # again no need of imputer because in categorical columns null values are not present
        ohe_pipe = Pipeline([
            ('ohe_encoder',OneHotEncoder(drop='first',handle_unknown = 'ignore',sparse_output = False))
        ])

        transformer = ColumnTransformer([
            # -----replaceble-----------------
            ('num',num_simple_pipe,num_cols),
            # -----------------------------------
            ('ohe',ohe_pipe,ord_cols)
        ])

        pipeline = Pipeline([
            ('preprocess',transformer),
            ('regressor',LinearRegression())
        ])


        # data spliting 
        from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV
        X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

        param = [
            # for Simple imputer
            {
                'preprocess__num':[num_simple_pipe],
                'preprocess__num__imputer__strategy':['mean','median']
            },

            # for KNN imputer
            {
                'preprocess__num':[num_knn_pipe],
                'preprocess__num__imputer__n_neighbors':[3,5,7]
            },

            # for iterative
            {
                'preprocess__num':[num_iter_pipe],
                'preprocess__num__imputer__estimator':[
                    LinearRegression(),
                    DecisionTreeRegressor(random_state = 42)
                ]
            }
        ]


        gs = GridSearchCV(
            estimator = pipeline,
            param_grid= param,
            cv = 5,
            scoring = 'r2',
            n_jobs = -1
        )

        gs.fit(X_train,y_train)

        print(gs.best_params_)
        print(gs.best_score_)

        model = gs.best_estimator_


        from sklearn.metrics import r2_score

        # trainig_r2_score
        y_train_pred = model.predict(X_train)
        train_r2 = r2_score(y_train,y_train_pred)


        # testing score
        y_test_pred = model.predict(X_test)
        test_r2 = r2_score(y_test,y_test_pred)


        mlflow.log_param('test_size',0.2)
        mlflow.log_param('cv',5)    
        mlflow.log_params(gs.best_params_)

        # log_metrics
        mlflow.log_metric('training r2_Score', train_r2 )
        mlflow.log_metric('testing_r2_score', test_r2)

        # log notebook
        mlflow.log_artifact('exp-2.ipynb')

        # logging data
        train_data = X_train
        train_data['price'] = y_train
        train_data = mlflow.data.from_pandas(train_data)

        test_data = X_test
        test_data['price'] = y_test
        test_data = mlflow.data.from_pandas(test_data)

        mlflow.log_input(train_data, context= 'Training data')
        mlflow.log_input(test_data, context='Testing data')


        # signature
        signature = mlflow.models.infer_signature(X_train, pipeline.predict(X_train))

        # log model 
        mlflow.sklearn.log_model(
                sk_model= model,
                name = 'Linear Regression',
                signature= signature
        )





        print('traning_score: ', train_r2)
        print('testing_score: ',test_r2)


{'preprocess__num': Pipeline(steps=[('imputer', KNNImputer()), ('scaler', RobustScaler())]), 'preprocess__num__imputer__n_neighbors': 7}
0.44233985911026324
traning_score:  0.46043443727441025
testing_score:  0.731435170839609
